In [1]:
import urllib.request; exec(urllib.request.urlopen('https://aic-data.aiffel.io/api/colab/setup?t=3vvc47bz').read().decode())

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



⏳  터널 준비 확인 중...

✅  터널 생성 완료!
🔗  URL: https://dsl-hostels-rhythm-forms.trycloudflare.com

아래 [URL 복사] 버튼을 누른 뒤 웹앱 연결창에 붙여넣으세요. (이 탭은 열어두세요)


✅ 웹앱에 자동 연결 요청을 보냈습니다. 잠시 후 웹앱 화면이 연결됩니다.


In [2]:
# ─────────────────────────────────────────────
# [C1] ◀ ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))
bikes = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    "bike_sharing.zip", "day.csv"))

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"자전거 대여 데이터: {bikes.shape[0]:,}행 × {bikes.shape[1]}열")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
내려받는 중… bike_sharing.zip
쇼핑 세션 데이터: 12,330행 × 18열
자전거 대여 데이터: 731행 × 16열

→ 준비 완료. 이제 여러분 차례입니다.


In [5]:
# [C2] ◀ 문제 1. 데이터 첫 대면 — 이 문제는 회귀인가, 분류인가
# ⌨️ 문제 1 — 데이터 구조·타겟·구매 전환율 확인

# 1) 앞 5행 + 열 정보
print("── 1) head() ──")
print(shoppers.head())

print("\n── 1) info() ──")
shoppers.info()

# 2) 타겟 값 종류와 개수
print("\n── 2) Revenue value_counts ──")
print(shoppers['Revenue'].value_counts())

# 3) 비율 — 두 가지 방법
print("\n── 3) 비율 (value_counts normalize) ──")
print(shoppers['Revenue'].value_counts(normalize=True))

print("\n── 3) 비율 (mean) ──")
print(shoppers['Revenue'].mean())

── 1) head() ──
   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               1                 0.000000   
1                     0.0               2                64.000000   
2                     0.0               1                 0.000000   
3                     0.0               2                 2.666667   
4                     0.0              10               627.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0         0.20       0.20         0.0         0.0   Feb                 1   
1         0.00       0.10         0.0     

**정리:**
- **결측치 없음** (전 열 12330 non-null)
- **dtype:** `object` 2개(`Month`, `VisitorType` — 나중에 인코딩 필요), `bool` 2개(`Weekend`, `Revenue`), 나머지는 수치형
- **타겟 개수:** False 10,422 / True 1,908 (총 12,330)
- **비율:** 15.47% — `value_counts(normalize=True)`와 `.mean()` 두 방법이 정확히 일치

In [6]:
# [C3] ◀ 문제 2. 분류 — 베이스라인부터 세운다
# ⌨️ 문제 2 — 베이스라인(Dummy) 정확도
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

# 1) X, y 준비
X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)

print("── 1) X, y 확인 ──")
print("X shape:", X.shape)
print("y 고유값:", y.unique())

# 2) train_test_split (stratify 필수)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("\n── 2) 분리 결과 ──")
print("train shape:", X_train.shape, "test shape:", X_test.shape)
print("train y 비율:\n", y_train.value_counts(normalize=True))
print("test y 비율:\n", y_test.value_counts(normalize=True))

# 3) DummyClassifier 학습 → 테스트 정확도
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_acc = dummy.score(X_test, y_test)

print("\n── 3) Dummy 베이스라인 정확도 ──")
print(f"{dummy_acc:.4f}")

── 1) X, y 확인 ──
X shape: (12330, 10)
y 고유값: [0 1]

── 2) 분리 결과 ──
train shape: (9864, 10) test shape: (2466, 10)
train y 비율:
 Revenue
0    0.845296
1    0.154704
Name: proportion, dtype: float64
test y 비율:
 Revenue
0    0.845093
1    0.154907
Name: proportion, dtype: float64

── 3) Dummy 베이스라인 정확도 ──
0.8451


**결과 확인:**
- X: 10개 수치형 열, 12,330행 → 9,864(train) : 2,466(test) = 정확히 8:2
- **stratify 검증 통과:** train 84.53%/15.47%, test 84.51%/15.49% — 원본 비율(84.53%/15.47%)이 양쪽 모두 거의 그대로 유지됐습니다
- **Dummy 베이스라인 정확도: 0.8451** — test set의 다수 클래스(0) 비율(0.845093)과 거의 일치. 당연한 결과입니다, Dummy는 무조건 0만 찍으니까요.

**이 숫자의 의미:** 이제부터 만들 어떤 모델이든 정확도가 **84.51%를 넘지 못하면 아무 의미가 없습니다.** 아무것도 학습하지 않은 모델과 다를 바 없다는 뜻이니까요. 반대로 "정확도 85%!"라는 결과를 봐도 이 베이스라인을 알기 전에는 그게 진짜 잘한 건지 판단할 수 없었을 겁니다 — 이게 바로 "성능은 항상 베이스라인 대비로 말한다"는 규칙의 실제 사례입니다.

In [7]:
# [C4] ◀ 문제 3. 분류 — 로지스틱 회귀로 베이스라인을 넘어라
# ⌨️ 문제 3 — 로지스틱 회귀 학습·평가 (베이스라인과 비교)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1) 스케일링 — 기준은 학습 데이터로만 잡는다
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # train: fit + transform
X_test_s = scaler.transform(X_test)         # test: transform만 (fit 다시 하면 안 됨)

# 2) 로지스틱 회귀 학습
logreg = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)

# 3) 세 값 비교
baseline_acc = dummy_acc  # 문제 2에서 구한 값
train_acc = logreg.score(X_train_s, y_train)
test_acc = logreg.score(X_test_s, y_test)

print("── 정확도 비교 ──")
print(f"베이스라인(Dummy) 정확도 : {baseline_acc:.4f}")
print(f"로지스틱 회귀 테스트 정확도 : {test_acc:.4f}")
print(f"로지스틱 회귀 학습 정확도 : {train_acc:.4f}")

print(f"\n베이스라인 대비 개선 : {(test_acc - baseline_acc)*100:.2f}%p")
print(f"학습-테스트 정확도 차이 : {(train_acc - test_acc)*100:.2f}%p")

── 정확도 비교 ──
베이스라인(Dummy) 정확도 : 0.8451
로지스틱 회귀 테스트 정확도 : 0.8796
로지스틱 회귀 학습 정확도 : 0.8839

베이스라인 대비 개선 : 3.45%p
학습-테스트 정확도 차이 : 0.44%p


**베이스라인 대비 개선: 3.45%p**
84.51% → 87.96%로, 아무것도 안 배운 Dummy보다 확실히 나아졌습니다. 수치형 10개 열만으로도(범주형 `Month`, `VisitorType`도 안 썼는데) 어느 정도 패턴을 잡아낸 겁니다. 다만 절대적인 크기로 보면 "3.45%p 개선"이 인상적인 숫자는 아닙니다 — 이 정도면 마케팅팀에 "많이 좋아졌다"고 말하기엔 다소 약한 수준이라, 다음 단계(범주형 피처 추가, 다른 모델 시도 등)로 더 끌어올릴 여지가 있다고 보는 게 맞습니다.

**과적합 신호: 거의 없음**
학습 정확도 88.39% vs 테스트 정확도 87.96%, 차이가 겨우 0.44%p입니다. 이 정도 차이는 과적합이라고 부르기엔 너무 작습니다. 오히려 "학습 데이터에서도 88%밖에 못 맞춘다"는 건, 모델이 데이터를 외운 게 아니라 실제로 일반화 가능한 패턴을 배웠다는 뜻이고, 학습/테스트 성능이 거의 같다는 건 모델이 안정적으로 작동한다는 좋은 신호입니다.

**다만 짚어볼 점 하나:** 정확도만으로는 이 모델이 "정말 쓸만한지" 아직 판단할 수 없습니다. 클래스가 84.5:15.5로 불균형하다는 걸 기억하시죠 — 테스트 정확도 87.96%가 실제로는 "소수 클래스(구매=True)를 얼마나 잘 잡아내는가"에 대해서는 아무 정보도 안 줍니다. 극단적으로, 모델이 True를 하나도 못 맞추고 그냥 애매한 경우만 몇 개 더 맞춰서 정확도를 올렸을 수도 있습니다.

In [8]:
# [C5] ◀ 문제 4. 회귀 — 하루 대여량을 예측하라
# ⌨️ 문제 4 — 자전거 대여량 회귀 (Dummy → LinearRegression)
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

FEATURES = ["temp", "atemp", "hum", "windspeed"]

# 1) X, y 준비
X_bike = bikes[FEATURES]
y_bike = bikes["cnt"]

print("── 1) X, y 확인 ──")
print("X shape:", X_bike.shape)
print("y 통계:\n", y_bike.describe())

# 2) train_test_split (stratify 없음)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bike, y_bike, test_size=0.2, random_state=RANDOM_STATE
)

print("\n── 2) 분리 결과 ──")
print("train shape:", X_train_b.shape, "test shape:", X_test_b.shape)

# 3) Dummy vs LinearRegression
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train_b, y_train_b)
dummy_r2 = dummy_reg.score(X_test_b, y_test_b)

linreg = LinearRegression()
linreg.fit(X_train_b, y_train_b)
linreg_r2 = linreg.score(X_test_b, y_test_b)

print("\n── 3) R² 비교 ──")
print(f"Dummy(평균 예측) R² : {dummy_r2:.4f}")
print(f"LinearRegression R² : {linreg_r2:.4f}")
print(f"\n베이스라인 대비 개선 : {linreg_r2 - dummy_r2:.4f}")

── 1) X, y 확인 ──
X shape: (731, 4)
y 통계:
 count     731.000000
mean     4504.348837
std      1937.211452
min        22.000000
25%      3152.000000
50%      4548.000000
75%      5956.000000
max      8714.000000
Name: cnt, dtype: float64

── 2) 분리 결과 ──
train shape: (584, 4) test shape: (147, 4)

── 3) R² 비교 ──
Dummy(평균 예측) R² : -0.0198
LinearRegression R² : 0.4995

베이스라인 대비 개선 : 0.5192


**Dummy R²가 정확히 0이 아니라 -0.0198인 이유:**
앞서 "Dummy R²는 0 근처"라고 했는데, 정확히 0이 되려면 test set의 평균이 train set의 평균과 똑같아야 합니다. `DummyRegressor(strategy="mean")`은 **train** 데이터의 평균을 외워서 예측값으로 쓰는데, R²를 계산할 때 비교 기준은 **test** 데이터 자신의 평균입니다. train 평균과 test 평균이 우연히 조금 다르니(샘플링에 따른 자연스러운 편차), "train 평균으로 test를 예측"하는 게 "test 평균으로 test를 예측"하는 것보다 살짝 못해서 R²가 0보다 약간 낮게(음수로) 나온 겁니다. 데이터가 731개뿐(자전거는 하루 단위라 2년치)이라 이런 편차가 더 잘 드러난 케이스입니다.

**LinearRegression R² = 0.4995:**
날씨 피처 4개(temp, atemp, hum, windspeed)만으로 하루 대여량 변동의 **약 50%**를 설명해낸다는 뜻입니다. Dummy 대비 개선폭 0.5192는 문제 3의 로지스틱 회귀 개선폭(3.45%p)과는 스케일이 다른 지표라 직접 비교는 안 되지만, "날씨 정보 자체가 대여량을 설명하는 데 꽤 중요한 변수"라는 걸 명확히 보여줍니다.

**마케팅/운영팀 관점에서 해석:** "날씨로 하루 대여량을 예측할 수 있을까?"라는 질문에 대해 — 절반은 설명되지만 나머지 절반은 날씨 말고 다른 요인(요일, 공휴일, 계절, 연도별 트렌드 등)에 달려 있다는 답이 됩니다.

In [9]:
# [C6] ◀ 문제 5. 첫 모델 카드 완성 — 오늘의 제출물
# ⌨️ 문제 5 — 두 모델을 하나의 표로

# LinearRegression의 학습 R²도 필요합니다 (지금까지는 테스트 R²만 구했음)
linreg_train_r2 = linreg.score(X_train_b, y_train_b)

model_card = pd.DataFrame([
    {
        "문제": "쇼핑 세션 구매 예측",
        "유형": "분류",
        "모델": "LogisticRegression",
        "주요 지표": "정확도",
        "베이스라인": round(baseline_acc, 4),
        "학습": round(train_acc, 4),
        "테스트": round(test_acc, 4),
    },
    {
        "문제": "자전거 하루 대여량 예측",
        "유형": "회귀",
        "모델": "LinearRegression",
        "주요 지표": "R²",
        "베이스라인": round(dummy_r2, 4),
        "학습": round(linreg_train_r2, 4),
        "테스트": round(linreg_r2, 4),
    },
])

print("── 모델 카드 v1 ──")
print(model_card.to_string(index=False))

── 모델 카드 v1 ──
           문제 유형                 모델 주요 지표   베이스라인     학습    테스트
  쇼핑 세션 구매 예측 분류 LogisticRegression   정확도  0.8451 0.8839 0.8796
자전거 하루 대여량 예측 회귀   LinearRegression    R² -0.0198 0.4503 0.4995


**자전거 모델의 학습 R²(0.4503) < 테스트 R²(0.4995)**

이게 보통 예상과는 반대 방향입니다. 일반적으로는 학습 성능이 테스트 성능보다 같거나 높은 게 정상인데(모델이 학습 데이터를 직접 보고 맞췄으니까요), 여기선 거꾸로 테스트가 더 잘 나왔습니다.

**왜 이런 일이 생길까:** 데이터가 731개뿐이고 그중 test는 147개밖에 안 됩니다. 이렇게 작은 데이터에서는 어떻게 나누느냐(어떤 147개가 test로 뽑히느냐)에 따라 우연히 "더 예측하기 쉬운" 조합이 test 쪽에 걸릴 수 있습니다. 이건 모델이 잘못됐다는 신호가 아니라, **데이터가 적을 때 train/test 분할 자체가 불안정하다**는 신호입니다. 앞서 Dummy R²가 정확히 0이 아니라 -0.0198이 나온 것과 같은 원인(train 평균 ≠ test 평균)의 연장선입니다.

**모델 카드에 반영할 문구:**
> "자전거 모델은 학습 R²(0.4503)가 테스트 R²(0.4995)보다 낮게 나왔다. 이는 과적합이 아니라 데이터 크기(731건)에서 오는 분할 불안정성으로 보이며, 신뢰구간을 확인하려면 단순 8:2 분할 대신 교차검증(cross-validation)이 필요하다."

이건 실무에서도 꽤 흔한 함정이라, "학습보다 테스트가 잘 나왔다고 무조건 좋아할 게 아니라 데이터 크기부터 확인한다"는 규칙 하나를 모델 카드 v1의 배운 점에 추가하시면 좋을 것 같습니다.

## 모델 카드 v1 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330 세션, 수치형 10개 열만 사용)
- 문제 유형: 분류
- 모델: 베이스라인(Dummy) → LogisticRegression
- 평가 지표 & 이유: 정확도(accuracy). 다만 타겟이 84.5% : 15.5%로 불균형해서, 정확도만으로는 "구매할 세션을 실제로 얼마나 잘 잡아내는지" 알 수 없다는 한계가 있음 — 다음 단계에서 precision/recall/F1으로 보완 필요
- 성능: 베이스라인 0.8451 → 내 모델 0.8796 / 학습 0.8839 vs 테스트 0.8796
- 과적합 진단: 학습-테스트 차이 0.44%p로 미미함 → 과적합 신호 없음. 모델이 데이터를 외운 게 아니라 일반화 가능한 패턴을 학습했다고 볼 수 있음
- 한계 & 다음 단계: (1) 범주형 8개 열(Month, VisitorType, OperatingSystems, Browser, Region, TrafficType, Weekend, 그리고 별도 인코딩이 필요한 열들)을 아직 안 써서 성능 개선 여지가 남아있음 — "피처 다루기" 시간에 인코딩 배운 뒤 추가 예정. (2) 클래스 불균형 때문에 정확도만으로 이 모델을 "쓸만하다"고 판단할 수 없음 — confusion matrix, precision/recall을 배우는 시간에 해결 예정. (3) 베이스라인 대비 개선폭(3.45%p)이 크지 않아, 다른 모델(트리 기반 등)이나 피처 추가로 개선 폭을 키울 필요가 있음

## 모델 카드 v1 — 자전거 대여량 예측

- 데이터: UCI Bike Sharing (731일, 날씨 피처 4개: temp, atemp, hum, windspeed)
- 문제 유형: 회귀
- 모델: 베이스라인(Dummy) → LinearRegression
- 평가 지표 & 이유: R²(설명력). 회귀에서는 정확도 개념이 없고, "평균만 찍는 것보다 얼마나 더 잘 설명하는가"를 보는 게 적합해서 R²를 씀
- 성능: 베이스라인 -0.0198 → 내 모델 0.4995 / 학습 0.4503 vs 테스트 0.4995
- 과적합 진단: 과적합 신호 없음 — 오히려 학습보다 테스트 R²가 더 높게 나옴. 이는 데이터가 731건으로 적어 8:2 분할 시 어떤 147개가 test로 뽑히느냐에 따라 결과가 흔들리는 "분할 불안정성"으로 보임(과대적합이 아니라 표본 크기 문제)
- 한계 & 다음 단계: (1) 날씨 4개 피처만으로 대여량 변동의 약 50%만 설명 — season, holiday, weekday, yr 같은 열을 추가하면 R²가 더 오를 가능성이 큼(다음 피처 확장 단계에서 시도). (2) 데이터가 731건뿐이라 단순 8:2 분할의 결과가 불안정함 — 교차검증(cross-validation)으로 여러 분할에서의 평균 R²와 변동폭을 확인하는 게 다음 단계로 필요함. (3) casual, registered는 타겟 유출 위험 때문에 처음부터 제외했고, 이후 다른 피처를 추가할 때도 "미래 시점에 실제로 알 수 있는 정보인지"를 계속 점검해야 함